In [1]:
import numpy as np 

from scipy.constants import c as c0
from scipy.interpolate import CubicSpline

In [2]:
import sys
from pathlib import Path

# Adding to path
PROJECT_ROOT = Path("/home/lsito/mast")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gui.meas_config_dialog import MeasurementConfigDialog
from src.data_models.meas_config import MeasurementConfig

from src.gui.structure_config_dialog import RFStructureLoaderDialog
from src.data_models.rf_structure import RFStructureParams

from src.core.config_parser import DataMixin

In [3]:
# In case you need to check the current working directory
from pathlib import Path
Path.cwd()

PosixPath('/home/lsito/mast/notebook')

In [17]:
# Parameter definition test

RF_params = RFStructureParams.from_json("../data/2021_03_18_-_TD31_Nx-raw_data/00-Config_Files/TD31_R1_CC.json")
Meas_params = MeasurementConfig()

print((RF_params.vg_))

[9008763.36289999 8756937.69818    8508709.542956   8264078.897228
 8023045.760996   7785610.13426    7551772.01702    7321531.409276
 7094888.311028   6871842.722276   6652394.64302    6436544.07326
 6224291.012996   6015635.462228   5810577.420956   5609116.88918
 5411253.8669     5216988.354116   5026320.350828   4839249.857036
 4655776.87274    4475901.39794    4299623.432636   4126942.976828
 3957860.030516   3792374.5937     3630486.66638    3472196.248556
 3317503.340228   3166407.941396   3018910.05206    2875009.67222
 2734706.801876  ]


In [18]:
# Target Frequency (Should be the frequency of the source) in Hz --> This is computed from the measurement config as target frequency to tune
f1 = 11.9942e9 
f1 = Meas_params.target_frequency_mhz * 1e6

# Resonant frequency of the bead-pull (written in the name of the file) --> Should be the frequency of the cavity
f0 = 11.993e9 
f = f0 # Just supoport variable

# Normalized frequency detuning for small detunings
    # Note: this is Eq. (3) however, it looks like the sign is opposite.
DeltaF = -2 * (f1-f0) / f0

# Group velocity interpolation
    # This is a list of group velocities for each cell, given in the json config file, apparently at the beginning of the cell (0, 1, 2, ..., noc-1)
    # we need vg at the middle of the cell (0.5, 1.5, ..., noc-0.5) for the calculation of Qe and beta, so we need to interpolate it with a cubic spline
    # 0.5   1   1.5   2   2.5   3   ...
    #  |    *    |    *    |    *
    #      vg1       vg2       vg3

vg = RF_params.vg_

    # Number of cells, given in the json config file
noc = RF_params.noc 

# Issue here
x = np.arange(1, noc+1)          # 1, 2, ..., noc # This is Matlab indexing
x_interp = np.arange(0.5, noc + 1 + 0.5) # 0.5, 1.5, ..., noc + 0.5

spline = CubicSpline(x, vg, bc_type="not-a-knot", extrapolate=True, axis=0)
vg_ = spline(x_interp)

# External Q approximated from Eq. (5)

    # This is the phase advance (NOT per cell), given in the json config file
phi0 = RF_params.phi0 
Q0 = RF_params.Q0_
Qe = c0 * phi0 / vg_

# Coupling coefficient beta of each cell from Wangler Chapt.5.5
# Qe:      Qe1      Qe2      Qe3      Qe4      Qe5
#           |        |        |        |        |
# 
# beta:   beta1    beta2    beta3    beta4
#         uses     uses     uses     uses
#         Qe1,Qe2  Qe2,Qe3  Qe3,Qe4  Qe4,Qe5
# Notice that here the cell is treated as a single input system; hence the power flowing out of the second 
# port is accounted for as power lost
# Hence Beta(i) = P_ext(1)/(P0(i)+P_ext(i+1)) = 1/Qe(i) / (1/Q0(i) + 1/Qe(i+1))

Q0 = np.array(Q0)
Qe_i = np.array(Qe[:-1])
Qe_ip1 = np.array(Qe[1:])

beta = (1/Qe_i) / (1/Q0 + 1/Qe_ip1)

# Reflection from the input port (from Eq. (1) with Q0 = Qe)

gamma_num = (beta-1)*(beta+1) - Qe_i**2 * DeltaF**2 - 1j*2*beta*Qe_i*DeltaF
gamma_den = (beta+1)**2 + Qe_i**2 * DeltaF**2
gamma = gamma_num / gamma_den

In [36]:
# Cumulative attenuation computation from Eq. (19) but already as an exponential term
v_particles = RF_params.v_particles

alpha = (v_particles * phi0) / (Q0 * vg)

att = np.ones(noc+1)

att[0] = 1.0
att[1:noc] = np.exp(-np.cumsum(alpha[:-1]))
att[-1] = att[-2] * np.exp(-alpha[-1])

## Up to now this was bbpare up to line 213 where bp_eval_routine gets called

What was up should still be in RF structure datamodel (as properties maybe)

Should be a routine to read and evaluate beadpull files

[1.         0.98654792 0.97293114 0.95914744 0.94519462 0.93107048
 0.91677288 0.90229969 0.88764885 0.87281834 0.85780622 0.84261065
 0.82722986 0.81166223 0.79590628 0.77996066 0.76382426 0.74749614
 0.73097563 0.71426235 0.69735624 0.6802576  0.66296716 0.6454861
 0.62781618 0.60995973 0.59191977 0.57370011 0.5553054  0.53674129
 0.51801451 0.49913302 0.48010619 0.46094489]
34
